# SimMIM Pre-training for SwinUnet Wildfire Model

Self-supervised pre-training using channel-group masking on fire-agnostic satellite data.
The model learns to reconstruct masked channel groups from visible ones, learning
cross-feature spatial relationships (e.g., predict weather from terrain).

**Prerequisites:**
- Pre-training TIFs in GCS bucket `lmudl-wildfire-compilation-bucket/WildfireSpreadTS/pretrain/`
- **Runtime -> Change runtime type -> GPU** (T4 or better)

**After pre-training:** load the saved encoder+decoder weights into SwinUnet and fine-tune on the fire dataset.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuration

In [ ]:
REPO_ORG   = "amindell11"
REPO_NAME  = "wildfire-ts-swin"
REPO_BRANCH = "feature/simim-pretrain"

GCS_BUCKET  = "lmudl-wildfire-compilation-bucket"
GCS_PREFIX  = "WildfireSpreadTS/pretrain"
LOCAL_TIF_DIR = "/content/pretrain_tifs"

OUTPUT_DIR  = "/content/drive/MyDrive/wildfire_runs/pretrain"

MAX_EPOCHS        = 200
BATCH_SIZE        = 32
BASE_LR           = 1.5e-4
WEIGHT_DECAY      = 0.05
WARMUP_EPOCHS     = 10
MASK_PROB         = 0.5
CROP_SIDE_LENGTH  = 128
STATS_YEARS       = (2020, 2021)
NUM_WORKERS       = 4
SEED              = 42

CHECKPOINT_INTERVAL = 20

## 3. Download TIFs from GCS to local disk

Copies all pre-training TIFs to Colab's local NVMe for fast I/O during training.

In [ ]:
!pip install -q gcsfs
from google.colab import auth
auth.authenticate_user()

import os
os.makedirs(LOCAL_TIF_DIR, exist_ok=True)

!gsutil -m cp -r gs://{GCS_BUCKET}/{GCS_PREFIX}/* {LOCAL_TIF_DIR}/

# Count downloaded TIFs
import glob
tifs = glob.glob(f"{LOCAL_TIF_DIR}/**/*.tif", recursive=True)
print(f"Downloaded {len(tifs)} TIF files")

## 4. Clone repo and install dependencies

In [ ]:
_repo_url = f"https://github.com/{REPO_ORG}/{REPO_NAME}.git"
!rm -rf /content/{REPO_NAME}
!git clone -b {REPO_BRANCH} {_repo_url} /content/{REPO_NAME}
!pip install -q -r /content/{REPO_NAME}/requirements.txt
!pip install -q rasterio
!git -C /content/{REPO_NAME} log --oneline -5

In [ ]:
import sys
sys.path.insert(0, f'/content/{REPO_NAME}')

## 5. Detect accelerator

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("WARNING: No GPU detected.")

## 6. Run Pre-training

In [ ]:
import os
import random
import types
import numpy as np
import torch
import torch.backends.cudnn as cudnn

from config import get_config
from networks.mae_swin_unet import SimMIMSwinUnet
from trainer_mae_pretrain import trainer_mae_pretrain

if torch.cuda.is_available():
    cudnn.benchmark = True
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Build config (reuse existing YAML for architecture params)
config_args = types.SimpleNamespace(
    cfg=f'/content/{REPO_NAME}/configs/swin_tiny_patch4_window4_128_wildfire.yaml',
    opts=['MODEL.SWIN.IN_CHANS', '40', 'MODEL.SWIN.N_TIMESTEPS', '1', 'MODEL.PRETRAIN_CKPT', 'None'],
    batch_size=None, zip=False, cache_mode='part', resume=None,
    accumulation_steps=None, use_checkpoint=False, amp_opt_level='',
    tag=None, eval=False, throughput=False,
)
config = get_config(config_args)

# Build model
model = SimMIMSwinUnet(
    config, img_size=CROP_SIDE_LENGTH, mask_prob=MASK_PROB,
    use_factored_embed=False,
).to(DEVICE)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Mask prob: {MASK_PROB}, 5 channel groups")

# Training args
args = types.SimpleNamespace(
    tif_dir=LOCAL_TIF_DIR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95),
    warmup_epochs=WARMUP_EPOCHS,
    num_workers=NUM_WORKERS,
    crop_side_length=CROP_SIDE_LENGTH,
    stats_years=STATS_YEARS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer_mae_pretrain(args, model, OUTPUT_DIR, device=DEVICE)

## 7. Fine-tune on fire dataset

Load the pre-trained encoder+decoder weights into a fresh SwinUnet(num_classes=2)
and fine-tune using the existing wildfire training pipeline.

Set `PRETRAIN_WEIGHTS` to the path of the saved pre-training weights, then run
the fire training notebook as usual.

In [ ]:
PRETRAIN_WEIGHTS = f"{OUTPUT_DIR}/pretrain_encoder_decoder.pth"

from networks.vision_transformer import SwinUnet

# Build fine-tuning model
net = SwinUnet(config, img_size=CROP_SIDE_LENGTH, num_classes=2,
               use_factored_embed=False).to(DEVICE)

# Load pre-trained weights
pretrained = torch.load(PRETRAIN_WEIGHTS, map_location=DEVICE)
model_dict = net.swin_unet.state_dict()
compatible = {k: v for k, v in pretrained.items()
              if k in model_dict and v.shape == model_dict[k].shape}
model_dict.update(compatible)
net.swin_unet.load_state_dict(model_dict)

print(f"Loaded {len(compatible)}/{len(model_dict)} pre-trained weights")
print(f"Skipped: {set(model_dict.keys()) - set(compatible.keys())}")
print("\nReady for fine-tuning. Use the train_and_test notebook with this model.")